## Compare the results of handcrafted, baselines, and HumemAI


In [1]:
from typing import Literal
import shutil
import os
from glob import glob
from humemai_research.utils import read_yaml
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from typing import Literal
import pandas as pd


def get_handcrafted(
    size: Literal["xxs", "xs", "s", "m", "l", "xl", "xxl"],
    base_path: str = "training-results/non-equal-object-probs/",
) -> pd.DataFrame:
    """Get hand-crafted results.

    room_size=xxs   num_obs=6.0     max_obs=6   min_obs=6
    room_size=xs    num_obs=6.52    max_obs=8   min_obs=5
    room_size=s     num_obs=5.64    max_obs=7   min_obs=5
    room_size=m     num_obs=6.3     max_obs=10  min_obs=5
    room_size=l     num_obs=5.32    max_obs=8   min_obs=5
    room_size=xl    num_obs=5.58    max_obs=7   min_obs=5
    room_size=xxl   num_obs=6.0     max_obs=8   min_obs=5

    Args:
        size: room size

    """

    df = pd.read_csv(
        os.path.join(
            base_path, f"handcrafted/hand-crafted-results-room_size={size}.csv"
        )
    )

    df["test_mean"] = pd.to_numeric(df["test_mean"], errors="coerce")
    df_sorted = df.sort_values(
        by=["long_capacity", "test_mean"], ascending=[True, False]
    )
    df = df_sorted.groupby("long_capacity").first().reset_index()

    # add column "room_size":
    df["room_size"] = size

    # Rename columns
    df.rename(
        columns={
            "long_capacity": "capacity",
            "test_mean": "test",
            "test_std": "std_test",
        },
        inplace=True,
    )

    df["val"] = np.nan
    df["agent_type"] = "handcrafted"
    df["#_runs"] = 5
    df["terminates_at"] = 99

    # Reorder columns in df1
    df = df[
        [
            "test",
            "std_test",
            "val",
            "#_runs",
            "capacity",
            "agent_type",
            "room_size",
            "explore_policy",
            "qa_function",
            "mm_policy",
        ]
    ]

    # rename columns
    df.rename(columns={"mm_policy": "mm"}, inplace=True)
    df.rename(columns={"qa_function": "qa"}, inplace=True)
    df.rename(columns={"explore_policy": "explore"}, inplace=True)

    return df


def determine_hyper_parameters(train: dict) -> dict:
    """Determine hyper parameters."""
    hyper_parameters = {}

    hyper_parameters["capacity"] = train["capacity"]["long"]
    hyper_parameters["room_size"] = train["env_config"]["room_size"]
    hyper_parameters["#_iter"] = train["num_iterations"]
    hyper_parameters["replay_size"] = train["replay_buffer_size"]
    hyper_parameters["terminates_at"] = train["env_config"]["terminates_at"]

    hyper_parameters["agent_type"] = "dqn"
    hyper_parameters["relu_between"] = train["dqn_params"]["relu_between_gcn_layers"]
    hyper_parameters["dropout_between"] = train["dqn_params"][
        "dropout_between_gcn_layers"
    ]
    hyper_parameters["num_layers"] = train["dqn_params"]["gcn_layer_params"][
        "num_layers"
    ]
    hyper_parameters["batch_size"] = train["batch_size"]
    hyper_parameters["emb_dim"] = train["dqn_params"]["gcn_layer_params"][
        "embedding_dim"
    ]
    hyper_parameters["triple_weight"] = train["dqn_params"]["gcn_layer_params"][
        "triple_qual_weight"
    ]
    if "intrinsic_explore_reward" in train:
        hyper_parameters["intrinsic"] = train["intrinsic_explore_reward"]

    if "mm_policy" in train:
        hyper_parameters["mm"] = train["mm_policy"]

    if "explore_policy" in train:
        hyper_parameters["explore"] = train["explore_policy"]
    hyper_parameters["gcn"] = train["dqn_params"]["gcn_layer_params"]["type"]
    if "scale_reward" in train:
        hyper_parameters["scale"] = train["scale_reward"]
    hyper_parameters["qa_function"] = train["qa_function"]

    if "use_raw_embeddings_for_qa" in train["dqn_params"]:
        hyper_parameters["use_raw_embeddings_for_qa"] = train["dqn_params"][
            "use_raw_embeddings_for_qa"
        ]

    if "learning_rate" in train:
        hyper_parameters["lr"] = train["learning_rate"]

    if "qa_entities" in train:
        hyper_parameters["qa_entities"] = train["qa_entities"]

    return hyper_parameters


def nanmean(data):
    return None if np.isnan(data).any() else round(np.mean(data))


def nanstd(data):
    return None if np.isnan(data).any() else round(np.std(data))


def nanmax(data):
    return None if np.isnan(data).any() else round(np.max(data))


def nanmin(data):
    return None if np.isnan(data).any() else round(np.min(data))


def get_dataframe(
    room_size: Literal["xxs", "xs", "s", "m", "l", "xl", "xxl"],
    base_path: str = "training-results/non-equal-object-probs/",
) -> pd.DataFrame:
    paths = glob(os.path.join(base_path, f"dqn/room_size={room_size}/*/*/results.yaml"))

    if len(paths) == 0:
        return pd.DataFrame()

    final = {}
    for result_path in tqdm(paths):
        result = read_yaml(result_path)
        train_path = result_path.replace("results.yaml", "train.yaml")
        train = read_yaml(train_path)

        val_score = max([foo["mean"] for foo in result["validation_score"]])
        test_score = result["test_score"]["mean"]

        hp = determine_hyper_parameters(train)

        hp_str = str(hp)
        if hp_str in final:
            final[hp_str]["val"].append(val_score)
            final[hp_str]["test"].append(test_score)
            final[hp_str]["path"].append(result_path.split("/")[5].split(".")[-1])

        else:
            final[hp_str] = {
                "val": [val_score],
                "test": [test_score],
                "hyper_parameters": hp,
                "path": [result_path.split("/")[5].split(".")[-1]],
            }

    df_list = []
    for key in final:
        data = final[key]
        hp = data["hyper_parameters"]
        df_list.append(
            {
                "test": nanmean(data["test"]),
                "std_test": nanstd(data["test"]),
                # "test_max": nanmax(data["test"]),
                # "test_min": nanmin(data["test"]),
                "val": nanmean(data["val"]),
                # "std_val": nanstd(data["val"]),
                "#_runs": len(data["test"]),
                "capacity": hp.get("capacity", None),
                "agent_type": hp.get("agent_type", None),
                "room_size": hp.get("room_size", None),
                "mm": hp.get("mm", "neural"),
                "qa": hp.get("qa_function", None),
                "explore": hp.get("explore", "neural"),
                # "#_iter": hp.get("#_iter", None),
                # "replay_size": hp.get("replay_size", None),
                # "warm_start": hp.get("warm_start", None),
                # "terminates_at": hp.get("terminates_at", None),
                # "target_update": hp.get("target_update", None),
                # "relu_between": hp.get("relu_between", None),
                # "dropout_between": hp.get("dropout_between", None),
                # "num_layers": hp.get("num_layers", None),
                # "batch_size": hp.get("batch_size", None),
                # "emb_dim": hp.get("emb_dim", None),
                # "triple_weight": hp.get("triple_weight", None),
                # "intrinsic": hp.get("intrinsic", None),
                # "lr": hp.get("lr", None),
                # "gcn": hp.get("gcn", None),
                # "scale": hp.get("scale", None),
                "use_raw_embeddings_for_qa": hp.get("use_raw_embeddings_for_qa", None),
                "qa_entities": hp.get("qa_entities", None),
                "path": data["path"],
            }
        )

    df = pd.DataFrame(df_list)
    df_sorted = df.sort_values(
        by=["capacity", "test"],
        ascending=[True, False],
    )
    return df_sorted


# Function to add blank rows and flag them
def add_blank_rows_and_flag(df):
    # Create a list to hold the new rows
    new_rows = []
    previous_capacity = None

    # Iterate through the dataframe rows
    for index, row in df.iterrows():
        if previous_capacity is not None and row["capacity"] != previous_capacity:
            # Add a blank row and flag it when the capacity changes
            blank_row = pd.Series({col: "" for col in df.columns})
            blank_row["flag"] = True
            new_rows.append(blank_row)
        # Append the current row
        new_row = row.copy()
        new_row["flag"] = False
        new_rows.append(new_row)
        previous_capacity = row["capacity"]

    # Create a new dataframe from the new rows
    new_df = pd.DataFrame(new_rows).reset_index(drop=True)
    return new_df


# Function to highlight the flagged rows
def highlight_blank_rows(row):
    if row.flag:
        return ["background-color: yellow"] * len(row)
    else:
        return [""] * len(row)


def get_all_data(
    size: Literal["xxs", "xs", "s", "m", "l", "xl", "xxl"],
    base_path: str = "training-results/non-equal-object-probs/",
    include_handcrafted: bool = True,
) -> pd.DataFrame:
    """Get all data."""

    if include_handcrafted:
        df_1 = get_handcrafted(
            size,
            base_path=base_path,
        )
        df_2 = get_dataframe(size, base_path=base_path)
        df = pd.concat([df_1, df_2], ignore_index=True)

        df = df.sort_values(
            by=["capacity", "test"],
            ascending=[True, False],
        )
    else:
        df = get_dataframe(size, base_path=base_path)

    df.rename(columns={"history_block_size": "history"}, inplace=True)

    # Add blank rows and flag them in the dataframe
    df_with_blanks = add_blank_rows_and_flag(df)

    # Apply the highlight function
    df_with_blanks_styled = df_with_blanks.style.apply(highlight_blank_rows, axis=1)

    # Apply number formatting to styled DataFrame
    df_with_blanks_styled = df_with_blanks_styled.format(na_rep="NaN", precision=4)

    return df, df_with_blanks_styled


def filter_paths(
    room_size: Literal["xxs", "xs", "s", "m", "l", "xl", "xxl"],
    agent_type: Literal["baseline", "episodic", "semantic", "hybrid"],
    num_iterations: int,
    capacity: int,
    base_path: str,
) -> list:
    paths = glob(
        os.path.join(base_path, f"baselines/room_size={room_size}/*/*/results.yaml")
    ) + glob(
        os.path.join(base_path, f"dqn/room_size={room_size}/*/*/explore/results.yaml")
    )

    if len(paths) == 0:
        return []

    filtered_paths = []
    for path in tqdm(paths):
        if "baseline" in path:
            train_path = path.replace("results.yaml", "train.yaml")
        else:
            train_path = path.replace("explore/results.yaml", "train.yaml")

        train_data = read_yaml(train_path)

        # Determine hyperparameters to check agent type and capacity
        hp = determine_hyper_parameters(train_data)

        if hp.get("#_iter") != num_iterations:
            continue

        # Check the agent type
        if hp.get("agent_type") != agent_type:
            continue

        # Check the capacity range
        if hp.get("capacity") != capacity:
            continue

        # If all conditions are met, add the path to the filtered list
        filtered_paths.append(path)

    return filtered_paths

/home/tk/.virtualenvs/agent-room-env-v2-qa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df, df_styled = get_all_data(
    "xl-different-prob",
    base_path="training-results/non-equal-object-probs/",
)
df_styled

100%|██████████| 156/156 [00:53<00:00,  2.90it/s]


,test,std_test,val,#_runs,capacity,agent_type,room_size,explore,qa,mm,use_raw_embeddings_for_qa,qa_entities,path,flag
0,58.8800,4.5967,NaN,5,2,handcrafted,xl-different-prob,avoid_walls,latest_strongest,handcrafted,NaN,NaN,NaN,False
1,,,,,,,,,,,,,,True
2,77.7200,3.8013,NaN,5,6,handcrafted,xl-different-prob,avoid_walls,latest_strongest,handcrafted,NaN,NaN,NaN,False
3,,,,,,,,,,,,,,True
4,262.0000,8.0000,295.0000,5,12,dqn,xl-different-prob,neural,llm,neural,NaN,NaN,"['165859', '985909', '485567', '866683', '351669']",False
5,216.0000,5.0000,247.0000,10,12,dqn,xl-different-prob,neural,bandit,neural,False,all,"['094227', '730038', '520522', '398245', '732490', '996265', '396022', '089097', '131127', '376721']",False
6,209.0000,9.0000,248.0000,5,12,dqn,xl-different-prob,neural,bandit,neural,False,room,"['718719', '005755', '291742', '977645', '190171']",False
7,194.0000,10.0000,236.0000,5,12,dqn,xl-different-prob,neural,bandit,neural,True,room,"['572900', '733276', '111825', '733772', '434576']",False
8,101.4800,7.9570,NaN,5,12,handcrafted,xl-different-prob,avoid_walls,latest_strongest,handcrafted,NaN,NaN,NaN,False
9,,,,,,,,,,,,,,True
